In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import ast

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder


from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import train_test_split
import shap

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
def describe_cluster(cluster_movies, n_genres=3, n_topics=3):
    main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
    top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(n_genres)
    genres_desc = ', '.join([g.replace('main_genre_', '') for g in top_genres.index])
    topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
    topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False)
    top_topics = [(int(i.split('_')[1]), topic_means[i]) for i in topic_means.head(n_topics).index]
    topic_desc = []
    for topic_num, _ in top_topics:
        top_words = ', '.join([feature_names[i] for i in nmf.components_[topic_num].argsort()[-4:][::-1]])
        topic_desc.append(f"Topic {topic_num}: {top_words}")
    top_decades = cluster_movies['release_decade'].value_counts().head(2)
    decades_desc = ', '.join(map(str, top_decades.index))

    top_year = cluster_movies['release_decade'].value_counts().head(2)
    year_desc = ', '.join(map(str, top_year.index))
    example_titles = ', '.join(cluster_movies['title'].head(3))
    return f"Genres: {genres_desc} | Decades: {decades_desc} | Year: {year_desc} | Top topics: {'; '.join(topic_desc)} | Example movies: {example_titles}"


In [ ]:
df_movies = pd.read_parquet("../data/movies_filtered_cleaned.parquet")
df_ratings = pd.read_parquet("../data/processed_df_ratings_filtered.parquet")
df_users = pd.read_parquet("../data/all_users_stats_post_movies_filter.parquet")

In [ ]:
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['overview'].fillna(''))

n_topics = 50
nmf = NMF(n_components=n_topics, random_state=0)
overview_topics = nmf.fit_transform(tfidf_matrix)
topic_cols = [f"topic_{i}" for i in range(n_topics)]
overview_topics_df = pd.DataFrame(overview_topics, columns=topic_cols)

df_movies = pd.concat([df_movies.reset_index(drop=True), overview_topics_df.reset_index(drop=True)], axis=1)

feature_names = tfidf.get_feature_names_out()
for idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]
    print(f"Topic {idx}: {', '.join(top_words)}")
    if idx > 5:
        break


In [ ]:
# df_nans = pd.DataFrame(df_movies.isna().sum()).reset_index(drop=False)
# df_nans.loc[df_nans[0] != 0]

In [ ]:
# df_movies["tag"]

In [ ]:
def robust_tags(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)   # convert arrays to list for later consistency
    elif pd.isna(x):
        return []
    else:
        return [str(x)]

df_movies['tag'] = df_movies['tag'].apply(robust_tags)


In [ ]:
df_movies["genre_list"].isna().sum()

In [ ]:
def extract_secondary_genres(genre_list):
    if isinstance(genre_list, (list, np.ndarray)):
        genre_list = list(genre_list)  # Ensure it's a list (not np.array)
        genres = genre_list[:2] + [None] * (2 - len(genre_list))
    elif genre_list is None:
        genres = [None, None]
    else:
        try:
            genres_eval = eval(genre_list) if isinstance(genre_list, str) and genre_list.startswith('[') else str(genre_list).split(',')
            genres_eval = [g.strip() for g in genres_eval if g.strip()]  # Clean whitespace
            genres = genres_eval[:2] + [None] * (2 - len(genres_eval))
        except Exception:
            genres = [None, None]
    return pd.Series(genres, index=['secondary_genre_1', 'secondary_genre_2'])

df_movies[['secondary_genre_1', 'secondary_genre_2']] = df_movies['genre_list'].apply(extract_secondary_genres)


In [ ]:
df_movies['genre_str'] = df_movies.apply(
    lambda row: ' '.join([
        str(g) for g in [row['secondary_genre_1'], row['secondary_genre_2']]
        if g is not None and str(g).strip().lower() != 'none'
    ]),
    axis=1
)

print("Empty genre_str rows:", (df_movies['genre_str'] == '').sum())

non_empty_genres = df_movies[df_movies['genre_str'].str.strip() != '']

genre_tfidf = TfidfVectorizer(max_features=20, stop_words=None)  # Fewer genres, so lower max_features
genre_tfidf_matrix = genre_tfidf.fit_transform(non_empty_genres['genre_str'])

n_genre_topics = min(10, genre_tfidf_matrix.shape[1])  # Never more topics than unique genres
nmf_genre = NMF(n_components=n_genre_topics, random_state=0)
genre_topics = nmf_genre.fit_transform(genre_tfidf_matrix)
genre_topic_cols = [f"genre_topic_{i}" for i in range(n_genre_topics)]
genre_topics_df = pd.DataFrame(genre_topics, columns=genre_topic_cols, index=non_empty_genres.index)

for col in genre_topic_cols:
    df_movies[col] = np.nan
df_movies.loc[non_empty_genres.index, genre_topic_cols] = genre_topics_df

# genre_feature_names = genre_tfidf.get_feature_names_out()
# for idx, topic in enumerate(nmf_genre.components_):
#     top_genres = [genre_feature_names[i] for i in topic.argsort()[-4:][::-1]]
#     print(f"Genre Topic {idx}: {', '.join(top_genres)}")
#     if idx > 5:
#         break

In [ ]:
df_movies['tag_str'] = df_movies['tag'].apply(
    lambda tags: ' '.join([str(tag) for tag in tags if tag is not None])
)

print("Empty tag_str rows:", (df_movies['tag_str'] == '').sum())

non_empty_tags = df_movies[df_movies['tag_str'].str.strip() != '']

tfidf = TfidfVectorizer(max_features=100, stop_words='english')

tfidf_matrix = tfidf.fit_transform(non_empty_tags['tag_str'])

n_topics = 50
nmf = NMF(n_components=n_topics, random_state=0)
tag_topics = nmf.fit_transform(tfidf_matrix)
tag_topic_cols = [f"tag_topic_{i}" for i in range(n_topics)]
tag_topics_df = pd.DataFrame(tag_topics, columns=tag_topic_cols)

df_movies = pd.concat([df_movies.reset_index(drop=True), tag_topics_df.reset_index(drop=True)], axis=1)

feature_names = tfidf.get_feature_names_out()
for idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]
    print(f"Topic {idx}: {', '.join(top_words)}")
    if idx > 5:
        break


In [ ]:
print("Total movies:", len(df_movies))
print("Non-empty tag_str rows:", (df_movies['tag_str'].str.strip() != '').sum())

In [ ]:
print(df_movies['tag'].head(10))

In [ ]:
df_movies.columns

In [ ]:
df_movies = pd.get_dummies(df_movies, columns=['main_genre'])

main_genre_cols = [col for col in df_movies.columns if col.startswith('main_genre_')]

features_for_cluster = (
    topic_cols + tag_topic_cols + main_genre_cols + genre_topic_cols +
    [
		'release_year',
		'release_month',
		'release_decade',
		'movie_age',
		'runtime',
		'popularity', 'vote_average',
		'vote_min',
		'vote_max',
		'vote_count',
		'popularity_score',
		'critical_success',
		'crowd_approval',
		'lead_actor_popularity',
		'director_popularity',
    ]
)

X = df_movies[features_for_cluster].fillna(0)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components='mle', random_state=0)
X_pca = pca.fit_transform(X_scaled)



In [ ]:
X.shape

In [ ]:
# X_scaled = StandardScaler().fit_transform(X)

# inertia = []
# # cluster_range = range(2, 251)  # Try between 2 and 30 clusters
# cluster_range = range(2, 151)  # Try between 2 and 30 clusters

# for k in cluster_range:
#     kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
#     kmeans.fit(X_scaled)
#     inertia.append(kmeans.inertia_)

# plt.figure(figsize=(14,10))
# plt.plot(cluster_range, inertia, marker='o')
# plt.xlabel('Number of clusters (k)')
# plt.ylabel('Inertia (within-cluster sum of squares)')
# plt.title('Elbow Method for Optimal k')
# plt.grid()
# plt.show()


In [ ]:
# Stop Code

In [ ]:
n_clusters = 60
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
df_movies['cluster'] = kmeans.fit_predict(X_pca)

In [ ]:
len(df_movies['cluster'].unique())

In [ ]:
for cluster_id in range(n_clusters):
	if cluster_id > 5:
		break
	print(f"\n=== Cluster {cluster_id} ===")
	cluster_movies = df_movies[df_movies['cluster'] == cluster_id]

	top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(5)
	print("Top genres:", ', '.join([g.replace('genre_', '') for g in top_genres.index]))

	print("Avg vote average:", cluster_movies['vote_average'].mean())
	print("Avg popularity score:", cluster_movies['popularity_score'].mean())
	# print("Release decades:", cluster_movies['release_decade'].value_counts().head(3).to_dict())

	# # Top topics (overview) in the cluster
	# topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False)
	# for i in topic_means.head(3).index:
	#     topic_idx = int(i.split('_')[-1])
	#     top_words = [feature_names[j] for j in nmf.components_[topic_idx].argsort()[-8:][::-1]]
	#     print(f"Top topic {topic_idx}: {', '.join(top_words)}")


In [ ]:
df_ratings = df_ratings.merge(df_movies[['movieId', 'cluster']], on='movieId', how='left')

user_cluster_scores = (
    df_ratings.groupby(['userId', 'cluster'])['rating']
    .mean()
    .reset_index()
    .rename(columns={'rating': 'cluster_mean_rating'})
)

user_id = 4
top_clusters = user_cluster_scores[user_cluster_scores['userId'] == user_id] \
    .sort_values('cluster_mean_rating', ascending=False)['cluster'].head(3).tolist()

seen_movies = df_ratings[df_ratings['userId'] == user_id]['movieId']
recommend_pool = df_movies[~df_movies['movieId'].isin(seen_movies) & df_movies['cluster'].isin(top_clusters)]
recommend_pool = recommend_pool.sort_values('popularity_score', ascending=False).head(10)  # Top 10

movie_row = recommend_pool.iloc[0]
cluster_id = movie_row['cluster']

cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
top_genres_count = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(3)
top_genres_names = [col.replace('main_genre_', '') for col in top_genres_count.index]

topic_cols = [col for col in df_movies.columns if col.startswith('topic_')]
feature_names = tfidf.get_feature_names_out()
top_words_for_topic = {
    idx: [feature_names[i] for i in nmf.components_[idx].argsort()[-8:][::-1]]
    for idx in range(nmf.components_.shape[0])
}
topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False)
top_topics = [(int(i.split('_')[1]), topic_means[i]) for i in topic_means.head(3).index]

user_ratings = df_ratings[df_ratings['userId'] == user_id].copy()
user_cluster_means = user_ratings.groupby('cluster')['rating'].mean()
user_affinity = user_cluster_means.get(cluster_id, None)

print(f"We recommend '{movie_row['title']}' because you tend to rate movies highly from cluster {cluster_id}.")
print("This cluster is mostly:")
print(f"- Genres: {', '.join(top_genres_names)}")
print(f"- Typical release decades: {', '.join(map(str, cluster_movies['release_decade'].value_counts().head(2).index))}")
print("Thematic topics include:")
for topic_number, _ in top_topics:
    print(f"  - Topic {topic_number}: {', '.join(top_words_for_topic[topic_number])}")
if user_affinity is not None:
    print(f"You give movies from this cluster an average rating of {user_affinity:.2f}.")


In [ ]:
movie_id = 1221
movie_row = df_movies[df_movies['movieId'] == movie_id]
cluster_id = int(movie_row['cluster'].iloc[0])

cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
top_genres_count = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(3)
top_genres_names = [col.replace('main_genre_', '') for col in top_genres_count.index]


top_topics = []
for i in topic_cols:
    mean_val = cluster_movies[i].mean()
    top_topics.append((i, mean_val))
top_topics = sorted(top_topics, key=lambda x: x[1], reverse=True)[:3]

In [ ]:
top_topics

# Viewing user n recommender settings


In [ ]:
def get_common_columns(df1: pd.DataFrame, df2: pd.DataFrame) -> list:
    """
    Finds and returns a list of column names that are common to both DataFrames.

    Args:
        df1: The first pandas DataFrame.
        df2: The second pandas DataFrame.

    Returns:
        A list of column names present in both DataFrames.
    """
    common_columns = list(set(df1.columns) & set(df2.columns))
    return common_columns
common_cols = get_common_columns(df_ratings, df_movies)


In [ ]:
user_rated = df_ratings[df_ratings['userId'] == user_id].merge(df_movies, on=common_cols)

In [ ]:
# # Get user's rated movies
# user_id = 4
# user_rated = df_ratings[df_ratings['userId'] == user_id].merge(df_movies, on=common_cols)

# plt.figure(figsize=(10, 5))
# top_rated = user_rated.sort_values('rating', ascending=False).head(10)
# sns.barplot(y=top_rated['title'], x=top_rated['rating'], palette='viridis')
# plt.title(f"Top 10 Movies Rated by User {user_id}")
# plt.xlabel("Rating")
# plt.ylabel("Movie Title")
# plt.show()

user_main_genre_counts = user_rated[main_genre_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
user_main_genre_counts.head(10).plot(kind='bar')
plt.title(f"Genres Most Rated by User {user_id}")
plt.ylabel("Count")
plt.xlabel("Genre")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(y=recommend_pool['title'], x=recommend_pool['popularity_score'], palette='magma')
plt.title(f"Top 10 Movie Recommendations for User {user_id} (by popularity)")
plt.xlabel("Popularity Score")
plt.ylabel("Movie Title")
plt.show()


In [ ]:
import plotly.express as px

fig = px.bar(
    recommend_pool,
    x='popularity_score',
    y='title',
    color='cluster',
    orientation='h',
    title=f"Recommended Movies for User {user_id} and Their Clusters"
)
fig.show()


# Why recommend


In [ ]:
plt.figure(figsize=(6, 6))
pd.Series(top_genres_count).plot.pie(autopct='%1.1f%%')
plt.title(f"Main Genres in Recommended Cluster {cluster_id}")
plt.ylabel("")
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
pd.Series(top_genres_count).plot.bar()
plt.title(f"Main Genres in Recommended Cluster {cluster_id}")
plt.ylabel("Number of Movies")
plt.xlabel("Genre")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Already computed: top_topics = [(topic_number, value), ...]
top_topics_for_bar = top_topics[:5]  # Show top 5
topic_labels = [f"Topic {n}" for n, v in top_topics_for_bar]
topic_scores = [v for n, v in top_topics_for_bar]

plt.figure(figsize=(8, 4))
sns.barplot(x=topic_labels, y=topic_scores)
plt.title(f"Top Topics in Cluster {cluster_id}")
plt.ylabel("Avg Topic Weight")
plt.xlabel("Topic")
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
user_cluster_means.plot(kind='bar')
plt.title(f"User {user_id} Average Ratings by Cluster")
plt.ylabel("Average Rating")
plt.xlabel("Cluster")
plt.show()


In [ ]:

for _, row in recommend_pool.iterrows():
    cluster_id = int(row['cluster'])
    cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
    description = describe_cluster(cluster_movies)
    print(f"\nWe recommend '{row['title']}' to you because:")
    print(description)


In [ ]:
main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(6)

plt.figure(figsize=(7, 4))
sns.barplot(x=top_genres.values, y=[g.replace('main_genre_', '') for g in top_genres.index])
plt.title("Top Genres in This Recommendation Cluster")
plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.show()

In [ ]:
topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False).head(5)

top_topic_labels = []
for i in topic_means.index:
    topic_idx = int(i.split('_')[1])
    top_words = ', '.join([feature_names[j] for j in nmf.components_[topic_idx].argsort()[-4:][::-1]])
    top_topic_labels.append(top_words)

plt.figure(figsize=(8, 4))
sns.barplot(x=topic_means.values, y=top_topic_labels)
plt.title("Top Topics/Themes in This Cluster")
plt.xlabel("Average Topic Strength")
plt.ylabel("Topic Keywords")
plt.show()


In [ ]:
topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False).head(5)

top_topic_labels = []
for i in topic_means.index:
    topic_idx = int(i.split('_')[1])
    top_words = ', '.join([feature_names[j] for j in nmf.components_[topic_idx].argsort()[-4:][::-1]])
    top_topic_labels.append(top_words)

plt.figure(figsize=(8, 4))
sns.barplot(x=topic_means.values, y=top_topic_labels)
plt.title("Top Topics/Themes in This Cluster")
plt.xlabel("Average Topic Strength")
plt.ylabel("Topic Keywords")
plt.show()


In [ ]:
decade_counts = cluster_movies['release_decade'].value_counts().sort_index()
plt.figure(figsize=(7, 4))
decade_counts.plot(kind='bar')
plt.title("Release Decades in This Cluster")
plt.xlabel("Decade")
plt.ylabel("Number of Movies")
plt.show()


In [ ]:
decade_counts = cluster_movies['release_decade'].value_counts().sort_index()
plt.figure(figsize=(7, 4))
decade_counts.plot(kind='bar')
plt.title("Release Decades in This Cluster")
plt.xlabel("Decade")
plt.ylabel("Number of Movies")
plt.show()


In [ ]:
display(cluster_movies[['title', 'release_year', 'popularity_score']].sort_values('popularity_score', ascending=False).head(5))


In [ ]:
plt.figure(figsize=(6, 1))
plt.barh(['Your Avg Rating'], [user_affinity], color='green')
plt.title("Your Average Rating for This Cluster")
plt.xlim(0, 10)
plt.show()


In [ ]:
def plot_cluster_dashboard(cluster_id, user_id=None, n_genres=6, n_topics=5, n_examples=5):
    # Grab all movies in the cluster
    cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
    if cluster_movies.empty:
        print("No movies found for this cluster.")
        return

    # --- Genres ---
    main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
    top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(n_genres)

    plt.figure(figsize=(14, 8))
    plt.subplot(2, 2, 1)
    sns.barplot(x=top_genres.values, y=[g.replace('main_genre_', '') for g in top_genres.index])
    plt.title("Top Genres")
    plt.xlabel("Number of Movies")
    plt.ylabel("Genre")

    # --- Topics ---
    topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
    topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False).head(n_topics)
    topic_labels = []
    for i in topic_means.index:
        topic_idx = int(i.split('_')[1])
        top_words = ', '.join([feature_names[j] for j in nmf.components_[topic_idx].argsort()[-4:][::-1]])
        topic_labels.append(top_words)

    plt.subplot(2, 2, 2)
    sns.barplot(x=topic_means.values, y=topic_labels)
    plt.title("Top Topics/Themes")
    plt.xlabel("Avg Topic Strength")
    plt.ylabel("Topic Keywords")

    # --- Release Decades ---
    plt.subplot(2, 2, 3)
    decade_counts = cluster_movies['release_decade'].value_counts().sort_index()
    sns.barplot(x=decade_counts.index.astype(str), y=decade_counts.values)
    plt.title("Release Decades")
    plt.xlabel("Decade")
    plt.ylabel("Number of Movies")

    # --- Example Movies Table ---
    example_movies = cluster_movies[['title', 'release_year', 'popularity_score']].sort_values(
        'popularity_score', ascending=False).head(n_examples)
    plt.subplot(2, 2, 4)
    plt.axis('off')
    table = plt.table(
        cellText=example_movies.values,
        colLabels=example_movies.columns,
        loc='center',
        cellLoc='center',
        colLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    plt.title("Sample Movies (Top by Popularity)", pad=20)

    plt.tight_layout(rect=[0, 0.05, 1, 0.97])
    plt.suptitle(f"Cluster {cluster_id} Dashboard", fontsize=18, fontweight='bold', y=1.04)
    plt.show()

    # --- User's Average Rating (if given) ---
    if user_id is not None:
        user_cluster_ratings = df_ratings[(df_ratings['userId'] == user_id) & (df_ratings['cluster'] == cluster_id)]
        if not user_cluster_ratings.empty:
            avg_rating = user_cluster_ratings['rating'].mean()
            print(f"User {user_id}'s average rating for movies in this cluster: {avg_rating:.2f}")
        else:
            print(f"User {user_id} has not rated movies in this cluster.")

# ---- Usage Example ----
# Show dashboard for the cluster of a recommended movie:
cluster_id = int(df_movies.iloc[0]['cluster'])  # change as needed
plot_cluster_dashboard(cluster_id, user_id=4)


In [ ]:
def plotly_cluster_dashboard(cluster_id, user_id=None, n_genres=6, n_topics=5, n_examples=5, n_user_genres=10):
    cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
    if cluster_movies.empty:
        print("No movies found for this cluster.")
        return

    main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
    top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(n_genres)
    print(top_genres)
    genre_labels = [g.replace('main_genre_', '') for g in top_genres.index][::-1]
    genre_counts = top_genres.values[::-1]

    topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
    topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False).head(n_topics)
    topic_numbers = [int(i.split('_')[1]) for i in topic_means.index]
    topic_scores = topic_means.values
    topic_keywords = [
        ', '.join([feature_names[j] for j in nmf.components_[num].argsort()[-4:][::-1]])
        for num in topic_numbers
    ]

    decade_counts = cluster_movies['release_decade'].value_counts().sort_index()
    decade_labels = [str(x) for x in decade_counts.index]
    decade_vals = decade_counts.values

    example_movies = cluster_movies[['title', 'release_year', 'popularity_score']].sort_values(
        'popularity_score', ascending=False).head(n_examples)
    example_table = go.Table(
        header=dict(
            values=["Title", "Year", "Popularity"],
            fill_color='paleturquoise',
            align='left'
        ),
        cells=dict(
            values=[example_movies['title'], example_movies['release_year'], example_movies['popularity_score'].round(2)],
            fill_color='lavender',
            align='left'
        )
    )

    rating_text = ""
    if user_id is not None:
        user_cluster_ratings = df_ratings[(df_ratings['userId'] == user_id) & (df_ratings['cluster'] == cluster_id)]
        if not user_cluster_ratings.empty:
            avg_rating = user_cluster_ratings['rating'].mean()
            rating_text = f"User {user_id}'s average rating for this cluster: {avg_rating:.2f}"
        else:
            rating_text = f"User {user_id} has not rated movies in this cluster."

    user_top_genres = None
    if user_id is not None:
        user_rated = df_ratings[df_ratings['userId'] == user_id].merge(df_movies, on='movieId')
        user_main_genre_cols = [col for col in df_movies.columns if col.startswith('main_genre_')]
        user_main_genre_counts = user_rated[user_main_genre_cols].sum().sort_values(ascending=False)
        user_main_genre_counts = user_main_genre_counts.head(n_user_genres)
        user_genre_labels = [g.replace('main_genre_', '') for g in user_main_genre_counts.index]
        user_genre_counts = user_main_genre_counts.values
    else:
        user_genre_labels, user_genre_counts = [], []

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Top Genres in Cluster", "Top Topics (keywords)",
            "Release Decades", "Sample Movies (Top by Popularity)",
            f"User {user_id}'s Top Genres" if user_id else "",
            ""
        ),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "table"}],
            [{"type": "bar"}, None]
        ],
        vertical_spacing=0.10
    )

    fig.add_trace(go.Bar(
		x=genre_counts, y=genre_labels, orientation='h',
		marker_color='rgba(63,81,181,0.8)', name="Cluster Genres"
	), row=1, col=1)

    fig.add_trace(go.Bar(
        x=topic_scores, y=[f"Topic {n}" for n in topic_numbers], orientation='h',
        marker_color='rgba(244,67,54,0.8)', name="Topics",
        hovertext=topic_keywords, hoverinfo="text+y"
    ), row=1, col=2)

    fig.add_trace(go.Bar(
        x=decade_labels, y=decade_vals,
        marker_color='rgba(76,175,80,0.8)', name="Decades"
    ), row=2, col=1)

    fig.add_trace(example_table, row=2, col=2)

    if user_genre_labels:
        fig.add_trace(go.Bar(
            x=user_genre_counts[::-1],  # To plot from most to least, top-down
            y=user_genre_labels[::-1],
            orientation='h',
            marker_color='rgba(0,150,136,0.8)',
            name="User's Top Genres"
        ), row=3, col=1)

    fig.update_layout(
        height=1100, width=1200,
        title_text=f"Cluster {cluster_id} Dashboard",
        showlegend=False,
        margin=dict(t=90, b=20),
    )

    if rating_text:
        fig.add_annotation(
            text=rating_text,
            xref="paper", yref="paper",
            x=0.5, y=1.18, showarrow=False,
            font=dict(size=15, color="black"),
            align="center"
        )

    fig.update_xaxes(title_text="Count", row=1, col=1)
    fig.update_xaxes(title_text="Avg Topic Strength", row=1, col=2)
    fig.update_xaxes(title_text="Decade", row=2, col=1)
    fig.update_xaxes(title_text="Number of Movies Rated", row=3, col=1)
    fig.update_yaxes(title_text="Genre", row=1, col=1)
    fig.update_yaxes(title_text="Topic", row=1, col=2)
    fig.update_yaxes(title_text="Number of Movies", row=2, col=1)
    fig.update_yaxes(title_text="Genre", row=3, col=1)
    fig.show()

plotly_cluster_dashboard(cluster_id=12, user_id=4)


In [ ]:
user_main_genre_counts = user_rated[main_genre_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
user_main_genre_counts.head(10).plot(kind='bar')
plt.title(f"Genres Most Rated by User {user_id}")
plt.ylabel("Count")
plt.xlabel("Genre")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# feature_cols = features_for_cluster
# model_data = df_ratings.merge(df_movies[['movieId'] + feature_cols], on='movieId', how='left')

# # Only use columns that actually exist
# actual_feature_cols = [col for col in feature_cols if col in model_data.columns]
# X = model_data[actual_feature_cols].fillna(0)
# y = model_data['rating']


In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# model = RandomForestRegressor(n_estimators=100, random_state=42)
# model.fit(X_train, y_train)

In [ ]:

# explainer = shap.TreeExplainer(model)
# # Pick a recommended movie for the user
# rec_movie_features = recommend_pool[feature_cols].iloc[0:1]  # first recommended movie

# shap_values = explainer.shap_values(rec_movie_features)

# # Visualize
# shap.initjs()
# shap.force_plot(explainer.expected_value, shap_values, rec_movie_features)

In [ ]:
# shap.plots.waterfall(shap.Explanation(
#     values=shap_values[0],
#     base_values=explainer.expected_value,
#     data=rec_movie_features.iloc[0].values,
#     feature_names=rec_movie_features.columns.tolist()
# ))


In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# # Train XGBoost
# model = xgb.XGBRegressor(n_estimators=100, max_depth=8, n_jobs=-1)
# model.fit(X_train, y_train)

# # Explain with SHAP
# explainer = shap.Explainer(model, X_train)
# shap_values = explainer(X_test[:10])
# shap.plots.waterfall(shap_values[0])

In [ ]:
# STOP CODE

In [ ]:
# for idx, row in recommend_pool.head(5).iterrows():
#     rec_movie_features = row[feature_cols].values.reshape(1, -1)
#     shap_values = explainer.shap_values(rec_movie_features)
#     print(f"\nExplanation for '{row['title']}':")
#     shap.plots.waterfall(shap.Explanation(
#         values=shap_values[0],
#         base_values=explainer.expected_value,
#         data=rec_movie_features[0],
#         feature_names=feature_cols
#     ))


In [ ]:

plt.figure(figsize=(10,5))
plt.bar(recommend_pool['title'], recommend_pool['popularity_score'])
plt.xticks(rotation=75, ha='right')
plt.title('Top Recommended Movies by Popularity Score')
plt.ylabel('Popularity Score')
plt.xlabel('Movie Title')
plt.tight_layout()
plt.show()


In [ ]:
main_genre_cols = [col for col in recommend_pool.columns if col.startswith('main_genre_')]
recommend_genres = recommend_pool[main_genre_cols].sum().sort_values(ascending=False)

nonzero_recommend_genres = recommend_genres[recommend_genres > 0]

plt.figure(figsize=(8,4))
sns.barplot(
    x=nonzero_recommend_genres.values,
    y=[g.replace('main_genre_', '') for g in nonzero_recommend_genres.index]
)
plt.title('Genre Distribution in Recommendations')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()


In [ ]:
main_genre_cols = [col for col in recommend_pool.columns if col.startswith('main_genre_')]
all_genre_names = [g.replace('main_genre_', '') for g in main_genre_cols]

all_counts = recommend_pool[main_genre_cols].sum().reindex(main_genre_cols, fill_value=0).values

fig = go.Figure(
    go.Barpolar(
        r=all_counts,
        theta=all_genre_names,
        marker_line_color="black",
        marker_line_width=2,
        opacity=0.8
    )
)
fig.update_layout(
    title="Genre Distribution in Recommendations (Radial Chart, All Genres)",
    polar=dict(
        radialaxis=dict(showticklabels=True, ticks=''),
    ),
    showlegend=False
)
fig.show()


In [ ]:
movie = recommend_pool.iloc[0]
cluster_id = movie['cluster']
cluster_avg = df_movies[df_movies['cluster'] == cluster_id][topic_cols].mean()

plt.figure(figsize=(10,3))
plt.plot(range(len(topic_cols)), [movie[col] for col in topic_cols], label='This Movie')
plt.plot(range(len(topic_cols)), cluster_avg.values, label='Cluster Avg')
plt.title('Topic Distribution: This Movie vs. Cluster Avg')
plt.xlabel('Topic')
plt.ylabel('Weight')
plt.legend()
plt.show()
